<a href="https://colab.research.google.com/github/Soheilp86/Statistical-Machine-Learning/blob/main/12-MLR_Performance_Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Performance Metrics for Multiple Linear Regression

**Objective**: Understand how to evaluate and compare multiple linear regression models using different performance metrics.

## 1. Setup: Libraries and Data

Let's start by importing our tools and creating a meaningful synthetic dataset.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm
from scipy import stats

# Set random seed for reproducibility
np.random.seed(42)

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

### Synthetic Dataset: House Price Prediction

We'll predict house prices based on:
- Square footage
- Number of bedrooms
- Age of the house

This is a realistic scenario where we expect these features to have a linear relationship with price.

In [4]:
# Generate synthetic data
n_samples = 100

# Features
square_footage = np.random.uniform(800, 4000, n_samples)  # sq ft
bedrooms = np.random.randint(1, 6, n_samples)              # number of bedrooms
age = np.random.uniform(0, 50, n_samples)                  # years old

# True relationship (with some noise)
# Price = 150 * sqft + 50000 * bedrooms - 500 * age + noise
price = (150 * square_footage +
         50000 * bedrooms -
         500 * age +
         np.random.normal(0, 20000, n_samples))  # Add realistic noise

# Create DataFrame
data = pd.DataFrame({
    'square_footage': square_footage,
    'bedrooms': bedrooms,
    'age': age,
    'price': price
})

print("Dataset Preview:")
print(data.head())
print(f"\nShape: {data.shape}")
print(f"\nBasic Statistics:")
print(data.describe())

Dataset Preview:
   square_footage  bedrooms        age          price
0     1998.528380         1  36.760806  300187.712617
1     3842.285781         4  10.453581  811123.038340
2     3142.380614         5  27.072399  683818.257969
3     2715.707149         4  34.789220  613989.071315
4     1299.259649         5  11.427501  442171.871194

Shape: (100, 4)

Basic Statistics:
       square_footage    bedrooms         age          price
count      100.000000  100.000000  100.000000     100.000000
mean      2304.578379    2.910000   25.794810  478135.568366
std        951.966115    1.400541   14.348628  170753.302585
min        817.670775    1.000000    0.259243  172957.431214
25%       1418.242434    1.750000   13.969405  325011.519738
50%       2285.255855    3.000000   24.827177  451683.668286
75%       3136.649981    4.000000   37.252879  627589.295656
max       3958.038197    5.000000   49.812685  815798.809667


---
## 2. Quick Reminder: RMSE and MSE

Before we dive into new metrics, let's quickly recall two metrics you already know:

### **Mean Squared Error (MSE = $(1/n) \sum (y_i -\hat y_i)^2$)**

Average of squared differences between predicted and actual values.
We use square root to get rid of the negative sign. So taking the average penalizes large errors heavily. But its hard to interpret directly as it does not have the same unit as y.


### **Root Mean Squared Error (RMSE = $\sqrt{\text{MSE}}$)**
Average prediction error" in the same units as y (e.g., dollars if y is price. Easier to interpret and communicate

These metrics measure **prediction accuracy**: how off your model's prediction is on average. But don't account for **model complexity** (how many variables we use for prediction) or how much **variance the model explains**.

In [5]:
# Fit the model using sklearn
X = data[['square_footage', 'bedrooms', 'age']]
y = data['price']

model = LinearRegression()
model.fit(X, y)

# Make predictions
y_pred = model.predict(X)

# Calculate MSE and RMSE
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y, y_pred)

print("=" * 60)
print("PREDICTION ERROR METRICS (Reminder)")
print("=" * 60)
print(f"MSE:  ${mse:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"MAE:  ${mae:,.2f}")
print("\nInterpretation:")
print(f"On average, our predictions are off by about ${rmse:,.0f}")

PREDICTION ERROR METRICS (Reminder)
MSE:  $521,301,041.08
RMSE: $22,832.02
MAE:  $18,318.65

Interpretation:
On average, our predictions are off by about $22,832


## 3. Coefficient of Determination

###$R^2$



$R^2$ is a statistical measure that shows how well a regression model predicts real-world data. It calculates the percentage of the variation in the dependent variable that can be explained by the independent variables in the model.


The formula for calculating R-squared is:

$$\frac{\text{explained variation}}{\text{total variation}}$$

Since the total variation in $y$ can be separated into two parts:

$$\text{Total variation = Explained variation + Unexplained variation}$$

we can compute $R^2$ by

$$ 1- \frac{\text{unexplained variation}}{\text{total variation}}$$


__How to compute explined and unexplianed variations?__

The total variation in target variable is:  

$$\text{total sum of squares (SST) } =\sum(y_i-\bar y)^2$$

It measures **How much the observed values of $y$ vary around their mean.** A large $SST$ means the observations are spread out. A small $SST$ means the observations are clustered close to the mean.


__Explained variation:__ Variation that our regression model explains:


$$\text{Model sum of squares (SSM)} =\sum_{i=1}^{n}(\hat y_i-\bar y)^2$$


It measures **How much the predicted values of $y$ vary around the mean (mean in y).** A large $SSM$ means predictions vary a lot from the mean, so the model explains a lot of variation. Small $SSM$ means the predictions are close to the mean, so the model explains little variation.



__Unexplained variation:__ Variation left unexplained by the model:

$$\text{Sum of Squares for Error (SSE)} =\sum_{i=1}^{n}(\hat y_i- y_i)^2$$

It measures how much the actual values are spread out around the predictions. Large SSE means actual values are far from the regression predictions, so the model makes larger errors. Small SSe means actual values are close to the predictions, so the model fits the data well.

### Example:



Suppose we use hours studied to predict exam score. The regression model gives us a predicted score for each student.

$$\text{Actual score:}\quad 60,\ 70,\ 70,\ 80,\ 90$$

and

$$\text{Predicted score:}\quad 60,\ 67,\ 74,\ 81,\ 88$$


__Step 1: Find the mean__

The mean of the actual scores is:

$$\bar y=\frac{60+70+70+80+90}{5}=74$$

So, 74 is the center of our data.

__Step 2: Total variation (SST)__

We measure how far each actual score is from the mean:

$$SST=\sum(y_i-\bar y)^2$$

$$=(60-74)^2+(70-74)^2+(70-74)^2+(80-74)^2+(90-74)^2$$

$$=196+16+16+36+256=520$$

So:

$$\boxed{SST=520}$$

This is the total variation in the exam scores.

__Step 3: Explained variation (SSM)__

Now we look at how far each predicted score is from the mean.

$$SSM=\sum(\hat y_i-\bar y)^2$$

$$=(60-74)^2+(67-74)^2+(74-74)^2+(81-74)^2+(88-74)^2$$

$$=196+49+0+49+196=490$$

So:

$$\boxed{SSM=490}$$

This is the variation explained by the regression model.

__Step 4: Unexplained variation (SSE)__

Now look at how far each actual score is from its prediction.

$$SSE=\sum(y_i-\hat y_i)^2$$

$$=(60-60)^2+(70-67)^2+(70-74)^2+(80-81)^2+(90-88)^2$$

$$=0+9+16+1+4=30$$

So:

$$\boxed{SSE=30}$$

This is the variation not explained by the model.

__Step 5: Put everything together__


$$SST=SSM+SSE$$

$$520=490+30$$

So we have:

$$\boxed{\text{Total variation}=\text{Explained variation}+\text{Unexplained variation}}$$

__Step 6: Calculate $R^2$__

$$R^2=\frac{SSM}{SST}$$

$$R^2=\frac{490}{520}\approx0.942$$

Therefore:

$$\boxed{R^2 \approx 94.2\%}$$

We can say:

The regression model explains approximately 94.2% of the variation in exam scores.

The remaining:

$$100%-94.2%=5.8%$$

is unexplained by the model.

So remember:

$SST$ → total variation around the mean
$SSM$ → explained variation around the mean
$SSE$ → unexplained variation around the predictions
$R^2$ → proportion of total variation explained by the model


###Adjusted $R^2$

A major limitation of $R^2$ is that it always increases when we add more variables, even if those variables are not useful.


Suppose we have a model with three predictors:

- hours studied
- attendance
- previous exam score
and:

$R^2=0.70$. W e might think:


"Great! The model explains 70% of the variation."
But now suppose we add another predictor:

favorite color

The new model might have:

$R^2=0.71$

Did the model improve?


Technically, yes. $R^2$ increased.
But did favorite color really provide useful information?
Maybe not.


**Adjusted $R^2$** addresses this problem. It measures how much variation the model explains while also penalizing the model for adding unnecessary variables. In other words, __Adjusted $R^2$ increases only when a new variable improves the model enough to justify its added complexity__.


$$
\boxed{
\text{Adjusted }R^2
=
1-\frac{(1-R^2)(n-1)}{n-p-1}
}
$$
where
$
n=\text{number of observations}
$,

$
p=\text{number of predictor variables}
$.








In [6]:
# Calculate R² using sklearn
r2 = r2_score(y, y_pred)

# Calculate Adjusted R² manually
n = len(y)
k = X.shape[1]  # number of predictors
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - k - 1)

print("=" * 60)
print("VARIANCE EXPLAINED METRICS")
print("=" * 60)
print(f"R²:          {r2:.4f}")
print(f"Adjusted R²: {adj_r2:.4f}")
print(f"\nInterpretation:")
print(f"- Our model explains {r2*100:.1f}% of the variance in house prices")
print(f"- After accounting for complexity (3 predictors), it's {adj_r2*100:.1f}%")
print(f"\nNote: Adjusted R² < R² (as expected due to penalty)")

VARIANCE EXPLAINED METRICS
R²:          0.9819
Adjusted R²: 0.9814

Interpretation:
- Our model explains 98.2% of the variance in house prices
- After accounting for complexity (3 predictors), it's 98.1%

Note: Adjusted R² < R² (as expected due to penalty)


## 4. F-Statistic: Is the Model overal Useful?



The F-test evaluates whether the **regression model as a whole** is useful for explaining $Y$. it tests whether at least one of your predictor variables actually helps explain the outcome.


### Step 1: State the hypotheses

For a multiple linear regression model $y = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$:

$$
H_0:\beta_1=\beta_2=\cdots=\beta_p=0
$$

$$
H_a:\text{At least one }\beta_j\neq0
$$

In words:

$$
H_0:\text{None of the predictors are useful.}
$$

$$
H_a:\text{At least one predictor is useful.}
$$


### Step 2: Look at the F-statistic

The F-statistic compares the variation explained by the model with the variation that remains unexplained.

$$
F=\frac{\text{variation explained by the model}}{\text{unexplained variation}}
$$

A larger $F$ means the model explains substantially more variation than what remains unexplained.

### Step 3: Look at the p-value

The **p-value** tells us whether the evidence against $H_0$ is strong enough at a level of significance (usually $5\%$).

$$
p\text{-value}<0.05
\rightarrow
\text{Reject }H_0 \quad \text{There is evidence that at least one predictor contributes to explaining variation in $Y$ at level of 5%}
$$


If:

$$
p\text{-value}\geq0.05
\rightarrow
\text{Fail to reject }H_0 \quad \text{There is not enough evidence that the model is useful overall at level of 5%}
$$


### Key point

$$
\boxed{\text{The F-test evaluates the model as a whole, not individual predictors.}}
$$

A significant F-test tells us that **at least one predictor is useful**, but it does not tell us which one.


In [ ]:
# Use statsmodels for detailed regression output
X_with_const = sm.add_constant(X)  # Add intercept
sm_model = sm.OLS(y, X_with_const).fit()

# Print summary (this is the standard output in statistics)
print(sm_model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.982
Model:                            OLS   Adj. R-squared:                  0.981
Method:                 Least Squares   F-statistic:                     1740.
Date:                Wed, 02 Sep 2026   Prob (F-statistic):           1.64e-83
Time:                        21:37:54   Log-Likelihood:                -1145.5
No. Observations:                 100   AIC:                             2299.
Df Residuals:                      96   BIC:                             2309.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const          -6115.6403   8503.394     -0.

### How to Read the Statsmodels Summary (Intuitive Guide)

The summary table looks overwhelming at first, but here's what students should focus on:

**Top section (right side)**:
- **R-squared**: The R² value we care about
- **Adj. R-squared**: The adjusted R² value (penalizes for adding predictors)
- **F-statistic** and **Prob (F-statistic)**: These answer "Is the model statistically significant?" If the Prob is < 0.05, our model is doing something meaningful, not just random noise.

**Middle section (the coefficient table)**:
- **coef**: The coefficient (slope) for each predictor. For example, square_footage has coef ≈ 150, meaning each additional square foot increases price by ~$150.
- **P>|t|**: The p-value for each predictor. If < 0.05, that predictor is statistically significant (meaningful).
- **[0.025, 0.975]**: The 95% confidence interval. If it includes 0, the predictor might not be significant.

For now, focus on the top section: **R², Adjusted R², and F-statistic p-value** tell you if your model is good and significant.

In [ ]:
# Extract F-statistic and p-value
f_stat = sm_model.fvalue
f_pval = sm_model.f_pvalue

print("=" * 60)
print("OVERALL MODEL SIGNIFICANCE (F-Statistic)")
print("=" * 60)
print(f"F-Statistic: {f_stat:.4f}")
print(f"P-value:     {f_pval:.2e}")
print()
if f_pval < 0.05:
    print("✓ Model is STATISTICALLY SIGNIFICANT (p < 0.05)")
    print("  → The predictors collectively explain variance better than baseline")
else:
    print("✗ Model is NOT statistically significant (p ≥ 0.05)")
    print("  → Predictors don't meaningfully improve over baseline")

OVERALL MODEL SIGNIFICANCE (F-Statistic)
F-Statistic: 1739.8820
P-value:     1.64e-83

✓ Model is STATISTICALLY SIGNIFICANT (p < 0.05)
  → The predictors collectively explain variance better than baseline


---
## 5. Summary: How to Choose and Interpret Metrics

### **Quick Reference Guide**

| Metric | What It Measures | When to Use | Interpretation |
|--------|------------------|-------------|----------------|
| **RMSE** | Prediction error (units of y) | Always | "Average error of ${RMSE:.0f}" |
| **MSE** | Squared prediction error | Less common (hard to interpret) | Compare models only |
| **R²** | % of variance explained | Always | "Model explains X% of variance" |
| **Adjusted R²** | R² with penalty for complexity | Compare models | "Fair comparison across #predictors" |
| **F-statistic** | Overall model significance | Always | p < 0.05 = statistically significant |

### **Typical Workflow**

1. **Fit your model** → Get predictions
2. **Check RMSE & R²** → "How well does it fit?"
3. **Check F-statistic** → "Is it statistically significant?"
4. **Compare models with Adjusted R²** → "Which is better?"

### **Red Flags**

- High R² but low F-statistic p-value → Something's wrong
- Adjusted R² much lower than R² → Too many predictors

In [ ]:
# Final Summary Table
print("\n" + "=" * 70)
print("COMPREHENSIVE MODEL EVALUATION SUMMARY")
print("=" * 70)
print()
print("Our Multiple Linear Regression Model:")
print(f"  Price ~ Square Footage + Bedrooms + Age")
print()
print("Metrics Summary:")
print("-" * 70)
print(f"RMSE:              ${rmse:>10,.0f}  (prediction error in dollars)")
print(f"R²:                {r2:>10.4f}  (explains {r2*100:.1f}% of variance)")
print(f"Adjusted R²:       {adj_r2:>10.4f}  (fair comparison metric)")
print(f"F-statistic:       {f_stat:>10.4f}  (p-value: {f_pval:.2e})")
print("-" * 70)
print()
print("Conclusions:")
print("✓ Model fits well (R² = {:.4f})".format(r2))
print("✓ Model is statistically significant (F-test p < 0.001)")
print("✓ Ready for prediction and inference!")
print("=" * 70)


COMPREHENSIVE MODEL EVALUATION SUMMARY

Our Multiple Linear Regression Model:
  Price ~ Square Footage + Bedrooms + Age

Metrics Summary:
----------------------------------------------------------------------
RMSE:              $    22,832  (prediction error in dollars)
R²:                    0.9819  (explains 98.2% of variance)
Adjusted R²:           0.9814  (fair comparison metric)
F-statistic:        1739.8820  (p-value: 1.64e-83)
----------------------------------------------------------------------

Conclusions:
✓ Model fits well (R² = 0.9819)
✓ Model is statistically significant (F-test p < 0.001)
✓ Ready for prediction and inference!
